In [1]:
import os
%pwd

'c:\\Users\\Sandeep\\Desktop\\Projects\\AI-heart-murmur-detection\\reseach'

In [2]:
os.chdir("../")
%pwd

'c:\\Users\\Sandeep\\Desktop\\Projects\\AI-heart-murmur-detection'

In [13]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen = True)
class DataTransformerConfig:
    root_dir: Path
    local_data_file: Path
    preprocessed_data_path: Path

In [14]:
from HeartBeat.constant import *
from HeartBeat.utils.common import read_yaml, create_directories

In [15]:
# 4 Update configuration manager

class configurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,     # Access to constants
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath) # read all config and params yaml files
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_transformer_config(self) -> DataTransformerConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformer_config = DataTransformerConfig(
            root_dir = config.root_dir,
            local_data_file = config.local_data_file,
            preprocessed_data_path = config.preprocessed_data_path
        )

        return data_transformer_config

In [16]:
from sklearn.model_selection import train_test_split
import torch
import numpy as np
import torch.nn.functional as F

In [17]:
# Components

class DataTransformer:
    def __init__(self, config: DataTransformerConfig):
        self.config = config

    def split_and_encode(self, x_data, y_data, test_y, classes, train_size=0.8, val_size=0.9, random_state=80085):
        """
        x_data:       feature array
        y_data:       label array
        test_y:       unlabeled test labels
        classes:      list of classes e.g. [0, 1, 2]
        train_size:   proportion for train split (default 0.8)
        val_size:     proportion of train set for validation (default 0.9)
        random_state: reproducibility seed
        """
        # Split
        X_train, X_test, y_train, y_test = train_test_split(
            x_data, y_data, train_size=train_size, random_state=random_state
        )
        X_train, X_val, y_train, y_val = train_test_split(
            X_train, y_train, train_size=val_size, random_state=random_state
        )

        # One hot encode using PyTorch
        num_classes = len(classes)

        def one_hot(labels, num_classes):
            labels = np.array(labels)
            
            # handle -1 unlabeled by temporarily replacing with 0
            mask = labels == -1
            labels_fixed = labels.copy()
            labels_fixed[mask] = 0
            
            t = torch.tensor(labels_fixed, dtype=torch.long)
            encoded = F.one_hot(t, num_classes=num_classes).numpy()
            
            # zero out the rows that were -1 (unlabeled)
            encoded[mask] = 0
            
            return encoded

        y_train = one_hot(y_train, num_classes)
        y_test  = one_hot(y_test,  num_classes)
        y_val   = one_hot(y_val,   num_classes)
        test_y  = one_hot(test_y,  num_classes)

        print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
        print(f"X_val:   {X_val.shape}   | y_val:   {y_val.shape}")
        print(f"X_test:  {X_test.shape}  | y_test:  {y_test.shape}")

        return X_train, X_val, X_test, y_train, y_val, y_test, test_y
    
    def load_preprocessed(self):
        load_path = self.config.preprocessed_data_path
        x_data = np.load(os.path.join(load_path, "x_data.npy"))
        y_data = np.load(os.path.join(load_path, "y_data.npy"))
        test_x = np.load(os.path.join(load_path, "test_x.npy"))
        test_y = np.load(os.path.join(load_path, "test_y.npy"))
        print("Preprocessed data loaded successfully")
        return x_data, y_data, test_x, test_y

    def save_transformed(self, X_train, X_val, X_test, y_train, y_val, y_test, test_y):
        save_path = self.config.root_dir
        os.makedirs(save_path, exist_ok=True)
        np.save(os.path.join(save_path, "X_train.npy"), X_train)
        np.save(os.path.join(save_path, "X_val.npy"),   X_val)
        np.save(os.path.join(save_path, "X_test.npy"),  X_test)
        np.save(os.path.join(save_path, "y_train.npy"), y_train)
        np.save(os.path.join(save_path, "y_val.npy"),   y_val)
        np.save(os.path.join(save_path, "y_test.npy"),  y_test)
        np.save(os.path.join(save_path, "test_y.npy"),  test_y)
        print(f"Transformed data saved to {save_path}")

In [20]:
# 7 Pipeline - Data Transformation
Classes = [0, 1, 2]

try:
    config = configurationManager()
    data_transformer_config = config.get_data_transformer_config()
    data_transformer = DataTransformer(config=data_transformer_config)

    save_path = data_transformer_config.local_data_file

    if all(os.path.exists(os.path.join(save_path, f)) for f in [
        "X_train.npy", "X_val.npy", "X_test.npy",
        "y_train.npy", "y_val.npy", "y_test.npy", "test_y.npy"
    ]):
        print("Transformed data already exists, skipping transformation...")

    else:
        print("Transformed data not found, running transformation...")

        # 1. Load preprocessed data
        x_data, y_data, test_x, test_y = data_transformer.load_preprocessed()

        # 2. Split and encode
        X_train, X_val, X_test, y_train, y_val, y_test, test_y = data_transformer.split_and_encode(
            x_data=x_data,
            y_data=y_data,
            test_y=test_y,
            classes=Classes
        )

        # 3. Save transformed data
        data_transformer.save_transformed(X_train, X_val, X_test, y_train, y_val, y_test, test_y)

except Exception as e:
    raise e

    


[2026-06-25 23:31:21,700: INFO: common: ymal file config\config.yaml loaded sucessfully]
[2026-06-25 23:31:21,704: INFO: common: ymal file params.yaml loaded sucessfully]
[2026-06-25 23:31:21,705: INFO: common: created directory at artifacts]
[2026-06-25 23:31:21,708: INFO: common: created directory at artifacts/data_transformation]


Transformed data not found, running transformation...
Preprocessed data loaded successfully
X_train: (1263, 52, 1) | y_train: (1263, 3)
X_val:   (141, 52, 1)   | y_val:   (141, 3)
X_test:  (351, 52, 1)  | y_test:  (351, 3)
Transformed data saved to artifacts/data_transformation
